# 06 Planner (dry-run only)

Generate a conservative move/keep review plan from the latest review outputs.

This notebook stays **dry-run only**. It proposes target paths where the evidence is strong and keeps ambiguous rows in a structured review queue.

Policy basis for `v2_4`:
- Company root is `{COMPANY_FOLDER}` with `CORPORATE`, `TEMPLATES`, `ARCHIVE`, and `ASSETS` beneath it.
- Active assets live under `{COMPANY_FOLDER}/ASSETS/{ASSET_FOLDER}` and archived assets under `{COMPANY_FOLDER}/ARCHIVE/{YEAR}/{ASSET_FOLDER}`.
- Filenames use `{TYPEID}_{PHASE}_{DOCTYPE}_{DESCRIPTION}_{DATE}_{VERSION}_{STATUS}.{EXT}` and path limits stay conservative for Windows.
- Exact-hash duplicates are marked for review, and `SUPERSEDED` may be kept in place or moved to `_SUPERSEDED`; this notebook therefore does **not** auto-route duplicates or superseded files unless you explicitly opt in to a local convention.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_5.yaml'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUTS_DIR  =', OUTPUTS_DIR)
print('POLICY_PATH  =', POLICY_PATH)


PROJECT_ROOT = c:\00_Developement\sch-file-organizer
OUTPUTS_DIR  = c:\00_Developement\sch-file-organizer\data\outputs
POLICY_PATH  = c:\00_Developement\sch-file-organizer\policy\SCH_fileserver_policy_v2_5.yaml


In [2]:
from src.policy_loader import PolicyLoader
from src.reporting import detect_latest_outputs, load_optional_parquet, build_review_frame
from src.planner import PlanConfig, build_plan, save_plan_outputs, plan_summary

policy = PolicyLoader.from_file(POLICY_PATH)
paths = detect_latest_outputs(OUTPUTS_DIR)

print(paths)

inv = load_optional_parquet(paths.inventory_path)
cls = load_optional_parquet(paths.classification_path)
txt = load_optional_parquet(paths.text_path)

review = build_review_frame(inventory_df=inv, classification_df=cls, text_df=txt)
print('Review rows:', len(review))

display(review[['relative_path', 'rule_status', 'text_status', 'rule_reason']].head(20))


ReviewPaths(inventory_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/inventory_Random_Files_WORKING_COPY_20260308_115612.parquet'), classification_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/rule_classification_20260308_130634.parquet'), text_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/inventory_with_text_20260308_144155.parquet'))
Review rows: 1806


,relative_path,rule_status,text_status,rule_reason
0,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
1,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
2,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
3,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
4,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
5,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
6,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
7,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
8,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy
9,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,ok,duplicate hash non-canonical copy


## Planner settings

Fill in `COMPANY_FOLDER` and `ASSET_FOLDER` when you are ready to produce full target paths.

Until then, the planner will still generate asset-relative targets and keep unresolved rows in review.

Leave `ENABLE_SUPERSEDED_FOLDER` and `ENABLE_DUPLICATE_FOLDER` as `False` unless you explicitly want to adopt a local folder convention beyond what `v2_4` defines.

In [5]:
COMPANY_FOLDER = "TEST-TST-123456789"      # e.g. 'SCH-...-123456789'
ASSET_FOLDER = "TST00p000-01_TEST_TEST"        # e.g. 'PVS15p473-01_PROJECT_LOCATION'
ROOT_MODE = 'ASSETS'       # 'ASSETS' or 'ARCHIVE'
ARCHIVE_YEAR = None        # integer only if ROOT_MODE == 'ARCHIVE'

ENABLE_SUPERSEDED_FOLDER = True
ENABLE_DUPLICATE_FOLDER = True
ENABLE_DEPRECATED_FOLDER = True

config = PlanConfig(
    company_folder=COMPANY_FOLDER,
    asset_folder=ASSET_FOLDER,
    root_mode=ROOT_MODE,
    archive_year=ARCHIVE_YEAR,
    enable_superseded_folder=ENABLE_SUPERSEDED_FOLDER,
    enable_duplicate_folder=ENABLE_DUPLICATE_FOLDER,
    enable_deprecated_folder=ENABLE_DEPRECATED_FOLDER,
)

config


PlanConfig(company_folder='TEST-TST-123456789', asset_folder='TST00p000-01_TEST_TEST', root_mode='ASSETS', archive_year=None, enable_superseded_folder=True, enable_duplicate_folder=True, enable_deprecated_folder=True, max_ready_confidence=0.9)

In [6]:
plan = build_plan(review, policy_loader=policy, config=config)
summary = plan_summary(plan)
summary


{'rows': 1806,
 'ready_rows': 130,
 'needs_user_input': 1676,
 'would_change_path': 130,
 'manual_review': 1676}

In [7]:
display(
    plan[[
        'relative_path',
        'rule_status',
        'planner_action',
        'planner_reason',
        'planner_confidence',
        'planner_target_relative_path',
        'planner_target_full_path',
    ]].head(25)
)

display(plan['planner_action'].value_counts(dropna=False).rename_axis('planner_action').reset_index(name='count'))


,relative_path,rule_status,planner_action,planner_reason,planner_confidence,planner_target_relative_path,planner_target_full_path
0,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
1,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
2,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
3,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
4,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
5,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
6,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
7,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
8,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...
9,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,move_to_duplicate_folder,duplicate exact-hash non-canonical copy; using...,0.9,_DUPLICATED/01_selected\random_1_dedup_target/...,TEST-TST-123456789\ASSETS\TST00p000-01_TEST_TE...


,planner_action,count
0,manual_review,1676
1,move_to_duplicate_folder,130


In [8]:
ready = plan[plan['planner_ready']].copy()
needs_input = plan[plan['planner_needs_user_input']].copy()
changes = plan[plan['planner_would_change_path']].copy()

print('Ready rows:', len(ready))
print('Needs input:', len(needs_input))
print('Would change path:', len(changes))

display(ready[['relative_path', 'planner_action', 'planner_target_relative_path']].head(20))
display(needs_input[['relative_path', 'planner_action', 'planner_reason']].head(20))
display(changes[['relative_path', 'planner_target_relative_path']].head(20))


Ready rows: 130
Needs input: 1676
Would change path: 130


,relative_path,planner_action,planner_target_relative_path
0,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
1,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
2,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
3,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
4,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
5,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
6,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
7,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
8,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
9,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...


,relative_path,planner_action,planner_reason
130,01_selected\New Text Document.ogb,manual_review,needs classification or rename mapping
131,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,manual_review,needs classification or rename mapping
132,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,manual_review,needs classification or rename mapping
133,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,manual_review,needs classification or rename mapping
134,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,manual_review,needs classification or rename mapping
135,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,manual_review,needs classification or rename mapping
136,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,manual_review,needs classification or rename mapping
137,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,manual_review,needs classification or rename mapping
138,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,manual_review,needs classification or rename mapping
139,01_selected\doclaynet_pdf\doclaynet_pdf_0005_N...,manual_review,needs classification or rename mapping


,relative_path,planner_target_relative_path
0,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
1,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
2,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
3,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
4,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
5,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
6,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
7,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
8,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...
9,01_selected\random_1_dedup_target\doclaynet_pd...,_DUPLICATED/01_selected\random_1_dedup_target/...


In [9]:
STAMP = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUTS_DIR / f'plan_dry_run_{STAMP}'
csv_path, parquet_path = save_plan_outputs(plan, output_base)
print('Saved:')
print(' -', csv_path)
print(' -', parquet_path)


Saved:
 - c:\00_Developement\sch-file-organizer\data\outputs\plan_dry_run_20260308_150102.csv
 - c:\00_Developement\sch-file-organizer\data\outputs\plan_dry_run_20260308_150102.parquet


## Read this before the next step

Good signs:
- `move_to_policy_folder` rows with sensible lifecycle targets
- `keep_in_place` rows where compliant files already sit correctly
- `manual_review` and `review_special_folder_policy` rows concentrated in legacy/messy areas

What stays intentionally unresolved here:
- rename proposals for non-compliant filenames
- archive/delete execution
- duplicate/superseded routing unless you explicitly enable that convention

Once this notebook looks good, the next layer is an **execution manifest** notebook that still does not modify files, but prepares a rollback-ready CSV of actions.
